In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Fresh kernel — Cell 1
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [3]:
spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .getOrCreate()

print("Spark version:", spark.version)  # confirm JVM is alive

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [4]:
# Read raw tips data
df_read_tip_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_tip.json")

In [5]:
df_dim_user = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_user")

In [6]:
df_dim_business = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business")

In [7]:
df_dim_date = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_date")

In [8]:
df_dim_tips = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_tips")

In [9]:
df_clean_text = df_read_tip_data.withColumn(
    "text_clean",
    F.lower(F.trim(F.regexp_replace("text", r"\s+", " ")))
).drop(F.col("text"))

df_clean_text = df_clean_text.withColumn("text_clean", F.regexp_replace("text_clean", r"([a-z])\1{2,}", "$1$1"))
df_clean_text = df_clean_text.withColumn("text_clean", F.regexp_replace("text_clean", r"([!?.,*~-])\1{2,}", "$1"))

In [10]:
df_join_user_tips = df_clean_text.join(df_dim_user,on=df_clean_text["user_id"] == df_dim_user["User_id"],how="inner")\
                   .select(df_dim_user["User_pk_sk"],df_clean_text["*"]).drop(df_clean_text["user_id"])

In [20]:
df_join_user_tips_with_business = df_join_user_tips.join(df_dim_business , on = df_join_user_tips["business_id"] == df_dim_business["Business_id"] ,how="inner")\
                                  .select(df_join_user_tips["*"],df_dim_business["Business_pk_sk"]).drop(df_join_user_tips["business_id"])

In [13]:
df_join_date_with_dim_date = df_join_user_tips_with_business.join(df_dim_date , on = df_join_user_tips_with_business["date"] == df_dim_date["Date"] ,how="inner")\
                             .select(df_join_user_tips_with_business["*"] ,df_dim_date["Date_pk_sk"] ).drop(df_join_user_tips_with_business["date"])

In [22]:
df_get_text_pk  = df_join_date_with_dim_date.join(df_dim_tips , on = df_join_date_with_dim_date["text_clean"] == df_dim_tips["Text"] ,how="inner")\
                  .select(df_join_date_with_dim_date["*"] ,df_dim_tips["tips_pk_sk"] ).drop(df_join_date_with_dim_date["text_clean"])

In [23]:

df_valid_review_number = df_get_text_pk.withColumn("clean_compliment_count",F.when( ( F.col("compliment_count") >= 0 ) ,F.col("compliment_count")).otherwise(0) ).drop(F.col("compliment_count"))

In [42]:
df_valid_review_number.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/fact_tips")

In [ ]:
df_valid_review_number.count()

In [ ]:
df_valid_review_number.show()

In [31]:
df_get_text_pk.count()

889315

In [ ]:
df_dim_business.columns

In [32]:
890408- 889315

1093

In [33]:
df_join_user_tips_with_business.count()

890408

In [ ]:
df_join_user_tip.columns

['User_pk_sk', 'user_id', 'business_id', 'date', 'text', 'compliment_count']

In [ ]:
df_join_user_tip.show()

In [17]:
df_join_user_tip.count()

890408

In [19]:
908915- 890408


18507